---
# Import LiB

In [354]:
import pandas as pd
import random
from collections import defaultdict, Counter
import numpy as np
import csv
import os

---
# INIT GLOBAL CONST

In [355]:
POPULASI = 50
ITERATION = 1500
MUTATION_PROB = 0.8
CROSSOVER_PROB = 0.7 
TOURNAMENT_SIZE = 10
SLOT_PER_KELAS = 36
JUMLAH_KELAS = 27
FITNESS_EXPORT = "AVG-4.csv" # input user
RECORD_EXPORT = "AVG-4.csv" # input user

---
# IMPORT DATA

In [356]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')
wali_kelas_df = pd.read_csv('../dataset/wali_kelas.csv')

---
# DICT MAKER

In [357]:
# =========================================================
# Mapping hari -> id
# Digunakan agar proses komputasi lebih cepat dibanding
# menggunakan string nama hari secara langsung.
# =========================================================
hariId = {
    "Senin": 1,
    "Selasa": 2,
    "Rabu": 3,
    "Kamis": 4,
    "Jumat": 5,
}

# =========================================================
# Slot per hari
# Menghitung jumlah slot jam pelajaran untuk setiap hari
# berdasarkan data slot_df.
# Hasilnya akan digunakan saat evaluasi fitness dan
# pembentukan jadwal.
# =========================================================

# Hitung jumlah slot tiap hari dari dataframe slot
slotPerHariDict = slot_df['hari'].value_counts().to_dict()

# Ubah key dari nama hari menjadi id hari
slotPerHariDict = {hariId[k]: v for k, v in slotPerHariDict.items()}

# Disimpan dalam numpy array agar perhitungan evaluasi
# (loop, indexing) lebih cepat
slotPerHari = np.array([
    slotPerHariDict[1],
    slotPerHariDict[2],
    slotPerHariDict[3],
    slotPerHariDict[4],
    slotPerHariDict[5]
])

# =========================================================
# Kelas -> tingkatan
# Mapping setiap kelas dengan tingkatan (misal kelas X, XI, XII)
# Informasi ini penting karena beberapa guru hanya mengajar
# mapel pada tingkatan tertentu.
# =========================================================
kelasTingkatan = dict(
    zip(kelas_df['kelas_id'], kelas_df['tingkatan'])
)

# Ambil seluruh id kelas dan urutkan
kelasIds = np.array(sorted(kelasTingkatan.keys()))

# Jumlah total kelas dalam sistem
JUMLAH_KELAS = len(kelasIds)

# =========================================================
# Mapel -> jam per minggu
# Menyimpan jumlah jam pelajaran setiap mapel per minggu
# =========================================================
jamPerMingguMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['jam_per_minggu'])
)

# Array id mapel
mapelIds = np.array(list(jamPerMingguMapel.keys()))

# Array jumlah jam tiap mapel
jamMapel = np.array(list(jamPerMingguMapel.values()))

# =========================================================
# MGMP Mapel
# MGMP = waktu rapat guru mapel tertentu
# Mapel tidak boleh dijadwalkan pada hari MGMP tersebut
# =========================================================
mgmpMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['MGMP'])
)

# Konversi hari MGMP dari string menjadi id hari
mgmpMapel = {
    mapel_id: hariId[hari]
    for mapel_id, hari in mgmpMapel.items()
}

# =========================================================
# Batas slot siang
# Slot setelah batas ini dianggap slot siang.
# Beberapa mapel diusahakan tidak ditempatkan di slot siang.
# =========================================================
batasSiang = np.array([
    5,  # senin
    5,  # selasa
    4,  # rabu
    5,  # kamis
    4   # jumat
])

# =========================================================
# Batas MGMP per hari
# Jumlah maksimum mapel MGMP yang boleh muncul dalam
# satu hari tertentu.
# =========================================================
batasMGMP = np.array([
    2,
    2,
    2,
    2,
    1
])

# =========================================================
# Guru valid untuk mapel + tingkatan
# Membuat relasi guru yang boleh mengajar mapel tertentu
# pada tingkatan tertentu.
# key   = (mapel_id, tingkatan)
# value = list guru yang boleh mengajar
# =========================================================
guruValidMapel = defaultdict(list)

for row in relasi_guru_mapel_df.itertuples():

    # pasangan mapel dan tingkatan
    key = (row.mapel_id, row.tingkatan)

    # tambahkan guru yang memenuhi kriteria
    guruValidMapel[key].append(row.guru_id)

# ubah menjadi dictionary biasa
guruValidMapel = dict(guruValidMapel)

# =========================================================
# Guru mengajar durasi
# Menghitung total beban mengajar setiap guru
# digunakan untuk mengecek constraint durasi guru
# =========================================================
maxJamGuru = (
    relasi_guru_mapel_df
    .groupby("guru_id")["durasi"]
    .sum()
    .to_dict()
)

# =========================================================
# Wali kelas
# Mapping guru yang menjadi wali kelas untuk kelas tertentu.
# Digunakan sebagai constraint tambahan dalam penjadwalan.
# =========================================================
waliKelas = dict(
    zip(wali_kelas_df['guru_id'], wali_kelas_df['kelas_id'])
)

# =========================================================
# Slot -> hari
# Setiap slot memiliki informasi hari ke berapa.
# Digunakan untuk mengetahui posisi hari dari setiap gen
# dalam kromosom.
# =========================================================
slotHari = np.array([
    hariId[row.hari]
    for row in slot_df.itertuples()
])

# Jumlah slot untuk setiap kelas
SLOT_PER_KELAS = len(slotHari)

# =========================================================
# Total gen kromosom
# Dalam representasi GA:
# kromosom = jadwal seluruh kelas
# gen      = satu slot pelajaran
# =========================================================
TOTAL_SLOT = JUMLAH_KELAS * SLOT_PER_KELAS

# =========================================================
# Fitness weight
# Bobot penalti setiap constraint dalam evaluasi fitness.
# Semakin besar bobotnya, semakin penting constraint tersebut.
# =========================================================
fitnessWeight = {
    "guruBentrok": 100,          # guru mengajar di dua kelas pada waktu sama
    "distribusiMapel": 20,       # distribusi mapel tidak merata
    "konsistensiGuruMapel": 20,  # mapel diajar guru berbeda
    "durasiGuru": 10,            # jam mengajar melebihi batas
    "waktuMGMP": 5,              # melanggar jadwal MGMP
    "mapelSiang": 5,             # mapel muncul di slot siang
    "cekWaliKelas": 5            # wali kelas tidak sesuai
}

# =========================================================
# Blok jam mata pelajaran
# Beberapa mapel harus diajarkan secara berurutan
# (misalnya 2 jam berturut-turut).
# Fungsi ini menentukan pembagian blok jam tersebut.
# =========================================================
def blokDistribusi(jam):

    # mapel 2 jam -> 1 blok 2 jam
    if jam == 2:
        return [2]

    # mapel 3 jam -> 1 blok 3 jam
    if jam == 3:
        return [3]

    # mapel 4 jam -> 2 blok (2+2)
    if jam == 4:
        return [2,2]

    # mapel 5 jam -> 2 blok (2+3)
    if jam == 5:
        return [2,3]

    # default
    return [jam]


# ======================================
# slot awal dan akhir hari
# Digunakan untuk mengetahui batas indeks
# slot setiap hari dalam kromosom.
# ======================================

# indeks slot pertama tiap hari
slotAwalHari = np.concatenate(([0], np.cumsum(slotPerHari)[:-1]))

# indeks slot terakhir tiap hari
slotAkhirHari = np.cumsum(slotPerHari)


# ======================================
# blok mapel
# Menyimpan pembagian blok jam untuk
# setiap mata pelajaran.
# ======================================

blokMapel = {}

for mapel_id, jam in jamPerMingguMapel.items():

    # menentukan blok distribusi jam mapel
    blokMapel[mapel_id] = blokDistribusi(jam)

---
# INDIVIDU CONSTRUCT

### Generate Per kelas


In [358]:
def generatePerKelas(tingkatan):

    # list sementara yang berisi blok-blok mapel
    # format: (mapel_id, guru_id, panjang_blok)
    blok_list = []

    # loop semua mata pelajaran yang ada di sistem
    for mapel_id, jam in jamPerMingguMapel.items():

        # ambil daftar guru yang valid untuk:
        # - mapel tersebut
        # - tingkatan kelas tertentu
        guruValid = guruValidMapel.get((mapel_id, tingkatan))

        # jika tidak ada guru yang bisa mengajar mapel ini
        # maka mapel tersebut dilewati
        if not guruValid:
            continue

        # pilih guru secara acak dari guru valid
        # randomisasi ini penting untuk membuat populasi awal GA bervariasi
        guru = random.choice(guruValid)

        # ambil distribusi blok jam mapel
        # contoh:
        # 4 jam -> [2,2]
        # 5 jam -> [2,3]
        blok = blokMapel[mapel_id]
    
        # setiap blok dimasukkan sebagai item dalam list
        for b in blok:
            blok_list.append((mapel_id, guru, b))

    # acak urutan blok mapel
    # agar posisi mapel dalam jadwal tidak selalu sama
    random.shuffle(blok_list)

    # array untuk menyimpan hasil jadwal
    mapel_arr = []
    guru_arr = []

    # konversi blok mapel menjadi slot-slot individu
    for mapel_id, guru, panjang in blok_list:

        # tambahkan mapel sebanyak panjang blok
        # misalnya blok = 2 -> [mapel,mapel]
        mapel_arr.extend([mapel_id]*panjang)

        # guru juga diulang sesuai panjang blok
        guru_arr.extend([guru]*panjang)

    # konversi ke numpy array agar operasi lebih cepat
    mapel_arr = np.array(mapel_arr, dtype=np.int16)
    guru_arr  = np.array(guru_arr, dtype=np.int16)

    # =====================================================
    # memastikan panjang jadwal sesuai jumlah slot kelas
    # =====================================================

    # jika jumlah slot melebihi slot tersedia
    if len(mapel_arr) > SLOT_PER_KELAS:

        # potong ke ukuran yang sesuai
        mapel_arr = mapel_arr[:SLOT_PER_KELAS]
        guru_arr  = guru_arr[:SLOT_PER_KELAS]

    # jika slot masih kurang
    while len(mapel_arr) < SLOT_PER_KELAS:

        # pilih mapel secara acak
        mapel = random.choice(mapelIds)

        # cek apakah ada guru yang bisa mengajar mapel tersebut
        guruValid = guruValidMapel.get((mapel, tingkatan))

        if guruValid:

            # pilih guru secara acak
            guru = random.choice(guruValid)

            # tambahkan ke slot jadwal
            mapel_arr = np.append(mapel_arr, mapel)
            guru_arr  = np.append(guru_arr, guru)

    # return jadwal untuk satu kelas
    return mapel_arr, guru_arr

### Membuat individu (1 jadwal penuh)

In [359]:
def individuConstruct():

    # Membuat matriks kosong untuk menyimpan jadwal mapel dan guru
    # Dimensi:
    # baris  = jumlah kelas
    # kolom  = jumlah slot waktu per kelas
    #
    # mapel_matrix -> menyimpan ID mata pelajaran pada setiap slot
    # guru_matrix  -> menyimpan ID guru yang mengajar pada slot tersebut
    mapel_matrix = np.zeros((JUMLAH_KELAS, SLOT_PER_KELAS), dtype=np.int16)
    guru_matrix  = np.zeros((JUMLAH_KELAS, SLOT_PER_KELAS), dtype=np.int16)

    # Loop setiap kelas yang ada di sistem
    # enumerate digunakan agar kita mendapatkan:
    # i        -> index baris pada matrix
    # kelas_id -> id kelas sebenarnya
    for i, kelas_id in enumerate(kelasIds):

        # Ambil tingkatan kelas
        # Contoh: kelas 10A -> tingkatan 10
        # Tingkatan ini digunakan untuk menentukan
        # guru yang boleh mengajar mapel tertentu
        tingkatan = kelasTingkatan[kelas_id]

        # Generate jadwal untuk satu kelas
        # fungsi ini akan menghasilkan:
        # mapel_row -> array mapel sepanjang SLOT_PER_KELAS
        # guru_row  -> array guru sepanjang SLOT_PER_KELAS
        mapel_row, guru_row = generatePerKelas(tingkatan)

        # Masukkan hasil jadwal tersebut ke baris matrix
        # sehingga setiap baris mewakili satu kelas
        mapel_matrix[i] = mapel_row
        guru_matrix[i]  = guru_row

    # Return satu individu jadwal lengkap
    # yang terdiri dari:
    # - matrix mapel
    # - matrix guru
    return mapel_matrix, guru_matrix

### Membuat Populasi

In [360]:
def generatePopulation(POPULASI):

    # Membuat array 3 dimensi untuk menyimpan populasi jadwal
    #
    # Dimensi:
    # POPULASI      -> jumlah individu dalam populasi GA
    # JUMLAH_KELAS  -> jumlah kelas dalam sistem
    # SLOT_PER_KELAS -> jumlah slot waktu per kelas
    #
    # pop_mapel menyimpan mata pelajaran
    # pop_guru menyimpan guru yang mengajar
    pop_mapel = np.zeros((POPULASI, JUMLAH_KELAS, SLOT_PER_KELAS), dtype=np.int16)
    pop_guru  = np.zeros((POPULASI, JUMLAH_KELAS, SLOT_PER_KELAS), dtype=np.int16)

    # Loop untuk membuat setiap individu dalam populasi
    for i in range(POPULASI):

        # Membuat satu individu jadwal lengkap
        # fungsi ini menghasilkan:
        # mapel_matrix -> jadwal mapel seluruh kelas
        # guru_matrix  -> jadwal guru seluruh kelas
        mapel_matrix, guru_matrix = individuConstruct()

        # Simpan individu ke dalam populasi
        # indeks i menunjukkan individu ke-i
        pop_mapel[i] = mapel_matrix
        pop_guru[i]  = guru_matrix

    # Return populasi awal
    # pop_mapel dan pop_guru berisi seluruh individu jadwal
    return pop_mapel, pop_guru

In [361]:
# Memanggil fungsi untuk membangkitkan populasi
pop_mapel, pop_guru = generatePopulation(POPULASI)

### Debug

In [362]:
kelas = 0
ind = 0

print("Mapel kelas", kelas+1)
print(pop_mapel[ind, kelas])

print("Guru kelas", kelas+1)
print(pop_guru[ind, kelas])

Mapel kelas 1
[ 5  5  6  6  6  5  5  4  4  2  2  3  3  3 12 12 12 11 11  8  8  7  7  7
  1  1 13 13  4  4  3  3  9  9 10 10]
Guru kelas 1
[48 48 44 44 44 48 48 47 47 33 33 36 36 36 55 55 55 43 43  5  5 21 21 21
 39 39 53 53 47 47 36 36 38 38 47 47]


---
# EVAL

### Guru Bentrok

In [363]:
def guruBentrok(guru_matrix):

        # Variabel untuk menghitung total pelanggaran bentrok guru
        pelanggaran = 0

        # Loop setiap slot waktu
        # SLOT_PER_KELAS merepresentasikan jumlah slot waktu dalam satu hari/minggu
        # yang sama untuk semua kelas
        for slot in range(SLOT_PER_KELAS):

            # Ambil semua guru yang mengajar pada slot waktu yang sama
            # guru_matrix memiliki bentuk:
            # (JUMLAH_KELAS, SLOT_PER_KELAS)
            # guru_matrix[:, slot] berarti:
            # semua kelas pada slot waktu yang sama
            guruSlot = guru_matrix[:, slot]

            # Mengambil nilai guru yang unik
            # Jika tidak ada bentrok maka jumlah unik = jumlah kelas
            unik = np.unique(guruSlot)

            # Hitung jumlah bentrok
            # Contoh:
            # guruSlot = [3, 5, 3, 7]
            # artinya guru 3 mengajar di dua kelas pada waktu yang sama
            # unik = [3,5,7]
            # bentrok = 4 - 3 = 1
            pelanggaran += len(guruSlot) - len(unik)

        # Mengembalikan total jumlah konflik guru
        return pelanggaran

### Durasi GUru mengajar

In [364]:
def durasiGuru(guru_matrix, maxJamGuru):

    # Variabel untuk menghitung total pelanggaran
    # Pelanggaran terjadi jika total jam mengajar guru
    # melebihi batas maksimal yang diperbolehkan
    pelanggaran = 0

    # Mengubah matrix guru menjadi array 1 dimensi
    # sehingga seluruh slot dari semua kelas digabung
    # contoh:
    # guru_matrix.shape = (10 kelas, 36 slot)
    # flatten() -> array 360 slot
    semuaGuru = guru_matrix.flatten()

    # Menghitung jumlah kemunculan setiap guru
    # guru_ids -> daftar guru unik
    # counts   -> jumlah slot yang diajar oleh guru tersebut
    # contoh:
    # guru_ids = [1,2,3]
    # counts   = [40,35,28]
    guru_ids, counts = np.unique(semuaGuru, return_counts=True)

    # Loop setiap guru yang muncul dalam jadwal
    for guru, jam in zip(guru_ids, counts):

        # Ambil batas maksimal jam mengajar guru
        # dari dictionary maxJamGuru
        # jika guru tidak ditemukan maka default = 0
        max_jam = maxJamGuru.get(guru, 0)

        # Jika jam mengajar melebihi batas
        if jam > max_jam:

            # Tambahkan selisihnya sebagai pelanggaran
            # contoh:
            # jam = 30
            # max_jam = 24
            # pelanggaran = 6
            pelanggaran += (jam - max_jam)

    # Return total pelanggaran durasi guru
    return pelanggaran

### Distribusi Mapel

In [365]:
def hitungBlok(slots):

    # Mengurutkan slot agar mudah mendeteksi slot yang berurutan
    # contoh: [5,2,3] -> [2,3,5]
    slots = sorted(slots)

    # list untuk menyimpan panjang setiap blok berurutan
    blok = []

    # panjang blok saat ini
    panjang = 1

    # loop dari slot kedua sampai terakhir
    for i in range(1,len(slots)):

        # jika slot sekarang berurutan dengan slot sebelumnya
        # contoh: 3 setelah 2
        if slots[i] == slots[i-1] + 1:
            panjang += 1

        else:
            # jika tidak berurutan maka blok sebelumnya selesai
            blok.append(panjang)

            # mulai blok baru
            panjang = 1

    # masukkan blok terakhir
    blok.append(panjang)

    # contoh output:
    # slots = [2,3,5,6,7]
    # blok  = [2,3]
    return blok



def distribusiMapel(mapel_matrix):

    # menghitung total pelanggaran distribusi mapel
    pelanggaran = 0

    # loop setiap kelas
    for kelas in range(JUMLAH_KELAS):

        # ambil jadwal mapel untuk kelas tersebut
        jadwal = mapel_matrix[kelas]

        # dictionary untuk menyimpan slot setiap mapel
        # key   = mapel_id
        # value = daftar slot
        mapelSlot = defaultdict(list)

        # loop setiap slot pada jadwal
        for slot, mapel in enumerate(jadwal):

            # simpan slot kemunculan mapel
            mapelSlot[mapel].append(slot)

        # cek setiap mapel dalam kelas tersebut
        for mapel, slots in mapelSlot.items():

            # ambil distribusi blok yang diharapkan
            # contoh:
            # mapel 4 jam -> [2,2]
            blok_expected = blokMapel.get(mapel, [])

            # jika hanya 1 blok maka tidak perlu dicek distribusi
            # contoh: 2 jam -> [2]
            if len(blok_expected) <= 1:
                continue

            # hitung blok aktual dari jadwal
            # contoh:
            # slots = [1,2,5,6]
            # blok_real = [2,2]
            blok_real = hitungBlok(slots)

            # cek apakah distribusi blok sesuai
            # sorted digunakan agar urutan blok tidak masalah
            if sorted(blok_real) != sorted(blok_expected):

                # jika tidak sesuai maka pelanggaran bertambah
                pelanggaran += 1

            # cek distribusi hari
            # ambil hari dari setiap slot
            hariSet = set(slotHari[s] for s in slots)

            # jika jumlah hari kurang dari jumlah blok
            # berarti ada blok mapel yang berada di hari yang sama
            if len(hariSet) < len(blok_expected):

                # tambahkan pelanggaran
                pelanggaran += 1

    # return total pelanggaran distribusi mapel
    return pelanggaran

### Mapel PJOK

In [366]:
def mapelSiang(mapel_matrix):

    # ID mata pelajaran yang tidak diinginkan berada di slot siang
    # contoh: mapel tertentu seperti olahraga
    # mapel berat sering dihindari di jam terakhir
    TARGET_MAPEL = 8

    # variabel untuk menghitung total pelanggaran
    pelanggaran = 0

    # loop semua slot waktu dalam satu minggu
    for slot in range(SLOT_PER_KELAS):

        # ambil hari dari slot tersebut
        # slotHari menyimpan mapping slot -> hari
        hari = slotHari[slot]

        # ambil batas slot siang untuk hari tersebut
        # contoh:
        # senin batas = 5
        batas = batasSiang[hari-1]

        # menghitung slot keberapa dalam hari tersebut
        #
        # contoh:
        # slot global = 14
        # slot awal hari = 12
        # slotHariKe = 2
        slotHariKe = slot - slotAwalHari[hari-1]

        # jika slot masih sebelum batas siang
        # maka tidak perlu dicek
        if slotHariKe < batas:
            continue

        # jika sudah melewati batas siang
        # maka hitung jumlah TARGET_MAPEL pada slot tersebut
        # mapel_matrix[:, slot] -> semua kelas pada slot yang sama
        # contoh:
        # [8,3,8,5]
        # berarti ada 2 pelanggaran
        pelanggaran += np.sum(mapel_matrix[:, slot] == TARGET_MAPEL)

    # return total pelanggaran mapel siang
    return pelanggaran

### Waktu MGMP

In [367]:
def waktuMGMP(mapel_matrix):

    # Variabel untuk menghitung total pelanggaran
    pelanggaran = 0

    # Loop setiap kelas
    for kelas in range(JUMLAH_KELAS):

        # Ambil jadwal mapel untuk kelas tersebut
        jadwal = mapel_matrix[kelas]

        # Loop setiap mapel yang memiliki aturan MGMP
        # mgmpMapel berisi mapping:
        # mapel_id -> hari MGMP
        for mapel, hariMGMP in mgmpMapel.items():

            # Mengubah hari MGMP menjadi index array (0-based)
            hari_idx = hariMGMP - 1

            # Menentukan batas slot awal dan akhir hari tersebut
            # agar hanya slot di hari MGMP yang diperiksa
            start = slotAwalHari[hari_idx]
            end   = slotAkhirHari[hari_idx]

            # Menghitung berapa kali mapel tersebut muncul
            # pada hari MGMP untuk kelas ini
            jumlah = np.sum(jadwal[start:end] == mapel)

            # Jika jumlah kemunculan melebihi batas yang diperbolehkan
            if jumlah > batasMGMP[hari_idx]:

                # Tambahkan selisihnya sebagai pelanggaran
                pelanggaran += jumlah - batasMGMP[hari_idx]

    # Return total pelanggaran MGMP
    return pelanggaran

### Wali Kelas

In [368]:
def cekWaliKelas(guru_matrix):

    # Variabel untuk menghitung total pelanggaran
    pelanggaran = 0

    # Loop setiap pasangan wali kelas
    # waliKelas berisi mapping:
    # guru_id -> kelas_id
    for guru, kelas in waliKelas.items():

        # Mengecek apakah guru tersebut mengajar di kelas yang menjadi wali-nya
        # guru_matrix[kelas-1] mengambil seluruh slot pada kelas tersebut
        # kelas-1 digunakan karena index array dimulai dari 0
        # contoh:
        # kelas = 3 -> index = 2
        if guru not in guru_matrix[kelas-1]:

            # Jika guru wali tidak mengajar di kelasnya sama sekali
            # maka dianggap pelanggaran
            pelanggaran += 1

    # Return total pelanggaran wali kelas
    return pelanggaran

### Konsistensi GUru mapel

In [369]:
def konsistensiGuruMapel(mapel_matrix, guru_matrix):

    # Variabel untuk menghitung total pelanggaran
    pelanggaran = 0

    # Loop setiap kelas
    for kelas in range(JUMLAH_KELAS):

        # Ambil jadwal mapel dan guru untuk kelas tersebut
        jadwalMapel = mapel_matrix[kelas]
        jadwalGuru  = guru_matrix[kelas]

        # Ambil daftar mapel unik yang muncul di kelas tersebut
        # agar tidak mengecek mapel yang sama berulang kali
        for mapel in np.unique(jadwalMapel):

            # Ambil semua guru yang mengajar mapel tersebut
            # jadwalMapel == mapel -> menghasilkan mask boolean
            # kemudian digunakan untuk memilih guru yang mengajar mapel itu
            guruSet = np.unique(jadwalGuru[jadwalMapel == mapel])

            # Jika lebih dari satu guru mengajar mapel yang sama
            # maka terjadi pelanggaran konsistensi
            if len(guruSet) > 1:

                # Tambahkan selisih jumlah guru sebagai pelanggaran
                # contoh:
                # guruSet = [3,7]
                # pelanggaran = 1
                # guruSet = [3,7,8]
                # pelanggaran = 2
                pelanggaran += len(guruSet) - 1

    # Return total pelanggaran konsistensi guru-mapel
    return pelanggaran

### main eval

In [370]:
def evaluasiIndividu(mapel_matrix, guru_matrix, maxJamGuru):

    # Fungsi ini mengevaluasi satu individu jadwal (satu solusi)
    # dengan menghitung jumlah pelanggaran untuk setiap constraint.

    # mapel_matrix : matriks mata pelajaran
    # baris = kelas
    # kolom = slot waktu

    # guru_matrix : matriks guru yang mengajar
    # strukturnya sama dengan mapel_matrix

    # maxJamGuru : dictionary yang berisi batas maksimal jam mengajar tiap guru

    # Fungsi mengembalikan dictionary yang berisi jumlah pelanggaran
    # untuk setiap jenis constraint

    return {

        # Mengecek apakah ada guru yang mengajar di lebih dari satu kelas
        # pada slot waktu yang sama
        "guruBentrok": guruBentrok(guru_matrix),

        # Mengecek apakah distribusi jam mata pelajaran sesuai dengan
        # blok yang seharusnya (misalnya 2+1 jam atau 3 jam)
        "distribusiMapel": distribusiMapel(mapel_matrix),

        # Mengecek apakah mata pelajaran tertentu muncul di jam siang
        # yang seharusnya dihindari
        "mapelSiang": mapelSiang(mapel_matrix),

        # Mengecek apakah total jam mengajar guru melebihi batas maksimum
        "durasiGuru": durasiGuru(guru_matrix, maxJamGuru),

        # Mengecek aturan MGMP, misalnya jumlah maksimal mapel tertentu
        # dalam satu hari
        "waktuMGMP": waktuMGMP(mapel_matrix),

        # Mengecek apakah wali kelas mengajar di kelasnya sendiri
        "cekWaliKelas": cekWaliKelas(guru_matrix),

        # Mengecek apakah satu mapel dalam satu kelas diajar oleh lebih
        # dari satu guru (harusnya konsisten satu guru)
        "konsistensiGuruMapel": konsistensiGuruMapel(mapel_matrix, guru_matrix)
    }

### Looping setiap individu

In [371]:
def hitungFitness(hasilEvaluasi):

    # Variabel untuk menyimpan total nilai fitness
    # Fitness biasanya merepresentasikan total penalti dari semua pelanggaran
    fitness = 0

    # Melakukan iterasi pada setiap jenis constraint
    # hasilEvaluasi berupa dictionary seperti:
    # {
    #   "guruBentrok": 5,
    #   "distribusiMapel": 3,
    #   "mapelSiang": 1,
    # }

    for k, v in hasilEvaluasi.items():

        # k = nama constraint
        # v = jumlah pelanggaran constraint tersebut

        # fitnessWeight adalah dictionary berisi bobot tiap constraint
        # contoh:
        # fitnessWeight = {
        #     "guruBentrok": 100,
        #     "distribusiMapel": 20,
        #     "mapelSiang": 5,
        # }

        # Pelanggaran dikalikan dengan bobot constraint
        # lalu ditambahkan ke total fitness
        fitness += fitnessWeight[k] * v

    # Mengembalikan total fitness
    # Semakin kecil nilai fitness -> semakin baik solusi jadwal
    return fitness

### Menyimpan hasil eval tiap individu

In [372]:
def evaluasiPopulasi(pop_mapel, pop_guru, maxJamGuru):

    # List untuk menyimpan hasil evaluasi setiap individu dalam populasi
    hasilPopulasi = []

    # pop_mapel.shape[0] = jumlah individu dalam populasi
    # setiap individu adalah satu kandidat solusi jadwal
    for i in range(pop_mapel.shape[0]):

        # Mengambil jadwal mapel dari individu ke-i
        # bentuknya matrix: kelas × slot
        mapel_matrix = pop_mapel[i]

        # Mengambil jadwal guru dari individu ke-i
        # strukturnya sama dengan mapel_matrix
        guru_matrix  = pop_guru[i]

        # Mengevaluasi individu menggunakan semua constraint
        # hasilnya berupa dictionary jumlah pelanggaran
        evaluasi = evaluasiIndividu(mapel_matrix, guru_matrix, maxJamGuru)

        # Menghitung nilai fitness berdasarkan jumlah pelanggaran
        # dan bobot masing-masing constraint
        fitness = hitungFitness(evaluasi)

        # Menyimpan hasil evaluasi individu dalam bentuk dictionary
        hasilPopulasi.append({
            "fitness": fitness,     # nilai fitness individu
            "evaluasi": evaluasi,   # detail pelanggaran constraint
            "index": i              # posisi individu dalam populasi
        })

    # Mengembalikan seluruh hasil evaluasi populasi
    return hasilPopulasi

In [373]:
hasil = evaluasiPopulasi(pop_mapel, pop_guru, maxJamGuru)

### Output debug

In [374]:
# best = min(hasil, key=lambda x: x["fitness"])

# print("Best Fitness:", best["fitness"])
# print("Detail Evaluasi:")
# print(best["evaluasi"])

---
# GA

### Tournament

In [375]:
def tournamentSelection(pop_mapel, pop_guru, hasilPopulasi):

    # Mengambil beberapa individu secara acak dari populasi
    # jumlah individu yang diambil ditentukan oleh TOURNAMENT_SIZE
    # contoh: jika TOURNAMENT_SIZE = 3 maka akan dipilih 3 kandidat secara acak
    kandidat = random.sample(hasilPopulasi, TOURNAMENT_SIZE)

    # Dari kandidat tersebut dipilih individu dengan fitness terbaik
    # min digunakan karena dalam timetabling fitness yang lebih kecil lebih baik
    terbaik = min(kandidat, key=lambda x: x["fitness"])

    # Mengambil index individu terbaik dalam populasi
    idx = terbaik["index"]

    # Mengembalikan jadwal mapel dan guru dari individu terpilih
    # .copy() digunakan agar perubahan pada offspring tidak mengubah parent asli
    return pop_mapel[idx].copy(), pop_guru[idx].copy()

### Crossover

In [376]:
def crossover(parent1_mapel, parent1_guru, parent2_mapel, parent2_guru):

    # Menentukan titik pemotongan crossover berdasarkan jumlah kelas
    # split berada di antara 1 sampai JUMLAH_KELAS-1
    # artinya sebagian kelas diambil dari parent1 dan sisanya dari parent2
    split = random.randint(1, JUMLAH_KELAS - 1)

    # Membuat matriks anak dengan ukuran yang sama seperti parent
    # diinisialisasi dengan nilai 0
    child1_mapel = np.zeros_like(parent1_mapel)
    child1_guru  = np.zeros_like(parent1_guru)

    child2_mapel = np.zeros_like(parent1_mapel)
    child2_guru  = np.zeros_like(parent1_guru)

    # Membentuk child1:
    # kelas sebelum titik split diambil dari parent1
    child1_mapel[:split] = parent1_mapel[:split]

    # kelas setelah titik split diambil dari parent2
    child1_mapel[split:] = parent2_mapel[split:]

    # Guru mengikuti struktur yang sama dengan mapel
    child1_guru[:split] = parent1_guru[:split]
    child1_guru[split:] = parent2_guru[split:]

    # Membentuk child2:
    # kebalikan dari child1
    # kelas sebelum split dari parent2
    child2_mapel[:split] = parent2_mapel[:split]

    # kelas setelah split dari parent1
    child2_mapel[split:] = parent1_mapel[split:]

    # Guru mengikuti struktur yang sama dengan mapel
    child2_guru[:split] = parent2_guru[:split]
    child2_guru[split:] = parent1_guru[split:]

    # Mengembalikan dua anak hasil crossover
    return child1_mapel, child1_guru, child2_mapel, child2_guru

### Mutasi

In [377]:
def mutasi(mapel_matrix, guru_matrix):

    # Mengevaluasi individu untuk mengetahui jumlah pelanggaran tiap constraint
    evaluasi = evaluasiIndividu(mapel_matrix, guru_matrix, maxJamGuru)

    # Menentukan constraint yang memiliki pelanggaran paling besar
    # constraint ini akan menjadi target mutasi
    constraint = max(evaluasi, key=evaluasi.get)

    # Memilih kelas secara acak yang akan dimodifikasi
    kelas = random.randint(0, JUMLAH_KELAS - 1)

    # Jika pelanggaran terbesar berasal dari distribusi mapel
    if constraint == "distribusiMapel":

        # Memilih dua slot berurutan untuk ditukar
        slot1 = random.randint(0, SLOT_PER_KELAS - 2)
        slot2 = slot1 + 1

        # Menukar posisi mapel pada dua slot tersebut
        mapel_matrix[kelas,slot1], mapel_matrix[kelas,slot2] = \
            mapel_matrix[kelas,slot2], mapel_matrix[kelas,slot1]

        # Menukar guru yang mengajar pada slot yang sama
        guru_matrix[kelas,slot1], guru_matrix[kelas,slot2] = \
            guru_matrix[kelas,slot2], guru_matrix[kelas,slot1]

    # Jika pelanggaran terbesar adalah guru bentrok
    elif constraint == "guruBentrok":

        # Memilih slot acak pada kelas tersebut
        slot = random.randint(0, SLOT_PER_KELAS-1)

        # Mengambil mapel pada slot tersebut
        mapel = mapel_matrix[kelas,slot]

        # Menentukan tingkatan kelas (misalnya kelas 10,11,12)
        tingkatan = kelasTingkatan[kelasIds[kelas]]

        # Mengambil daftar guru yang valid untuk mapel dan tingkatan tersebut
        guruValid = guruValidMapel.get((mapel, tingkatan))

        # Jika ada guru yang valid
        if guruValid:

            # Mengganti guru dengan salah satu guru valid secara acak
            guru_matrix[kelas,slot] = random.choice(guruValid)

    # Jika constraint lain yang dominan
    else:

        # Memilih dua slot acak dalam satu kelas
        slot1 = random.randint(0, SLOT_PER_KELAS - 1)
        slot2 = random.randint(0, SLOT_PER_KELAS - 1)

        # Menukar posisi mapel antara dua slot tersebut
        mapel_matrix[kelas,slot1], mapel_matrix[kelas,slot2] = \
            mapel_matrix[kelas,slot2], mapel_matrix[kelas,slot1]

        # Menukar guru pada slot yang sama
        guru_matrix[kelas,slot1], guru_matrix[kelas,slot2] = \
            guru_matrix[kelas,slot2], guru_matrix[kelas,slot1]

    # Mengembalikan individu yang telah dimutasi
    return mapel_matrix, guru_matrix

### Repair

In [378]:
def repairOperator(mapel_matrix, guru_matrix):

    # Mengevaluasi individu untuk mengetahui jumlah pelanggaran constraint
    evaluasi = evaluasiIndividu(mapel_matrix, guru_matrix, maxJamGuru)

    # Repair hanya dilakukan jika terdapat pelanggaran guru bentrok
    if evaluasi["guruBentrok"] > 0:

        # Iterasi setiap slot waktu
        for slot in range(SLOT_PER_KELAS):

            # Mengambil semua guru yang mengajar pada slot tersebut
            # (dari semua kelas pada waktu yang sama)
            guruSlot = guru_matrix[:,slot]

            # Mengambil guru unik dan jumlah kemunculannya
            unik, counts = np.unique(guruSlot, return_counts=True)

            # Mendeteksi guru yang muncul lebih dari sekali pada slot yang sama
            # artinya guru tersebut mengajar di lebih dari satu kelas
            bentrok = unik[counts > 1]

            # Memproses setiap guru yang mengalami bentrok
            for g in bentrok:

                # Mencari kelas mana saja yang menggunakan guru tersebut
                kelasBentrok = np.where(guruSlot == g)[0]

                # Memilih salah satu kelas secara acak untuk diperbaiki
                kelas = random.choice(kelasBentrok)

                # Mengambil mapel pada kelas dan slot tersebut
                mapel = mapel_matrix[kelas,slot]

                # Menentukan tingkatan kelas (misal kelas 10, 11, 12)
                tingkatan = kelasTingkatan[kelasIds[kelas]]

                # Mengambil daftar guru yang valid untuk mapel dan tingkatan ini
                guruValid = guruValidMapel.get((mapel, tingkatan))

                # Jika ada guru yang valid
                if guruValid:

                    # Mengganti guru yang bentrok dengan guru valid lain secara acak
                    guru_matrix[kelas,slot] = random.choice(guruValid)

    # Mengembalikan jadwal yang telah diperbaiki
    return mapel_matrix, guru_matrix

---
# GWO

In [379]:
def runGWO(pop_mapel, pop_guru, hasilPop, iter_gwo=0, max_iter=20):

    hasilSorted = sorted(hasilPop, key=lambda x: x["fitness"])

    alpha = hasilSorted[0]["index"]
    beta  = hasilSorted[1]["index"]
    delta = hasilSorted[2]["index"]

    alpha_mapel = pop_mapel[alpha]
    alpha_guru  = pop_guru[alpha]

    beta_mapel = pop_mapel[beta]
    beta_guru  = pop_guru[beta]

    delta_mapel = pop_mapel[delta]
    delta_guru  = pop_guru[delta]

    # parameter convergence GWO
    a = 2 * (1 - (iter_gwo / max_iter))

    new_pop_mapel = []
    new_pop_guru  = []

    for i in range(len(pop_mapel)):

        mapel = pop_mapel[i].copy()
        guru  = pop_guru[i].copy()

        # elitism
        if i == alpha:
            new_pop_mapel.append(mapel)
            new_pop_guru.append(guru)
            continue

        # ===============================
        # pilih beberapa kelas sekaligus
        # ===============================
        kelas_list = random.sample(range(JUMLAH_KELAS), k=3)

        for kelas in kelas_list:

            # pilih hari random
            hari = random.randint(0,4)

            start = slotAwalHari[hari]
            end   = slotAkhirHari[hari]

            # ukuran segmen
            panjang = end-start
            size = random.randint(1, max(2, panjang//2))
            s = random.randint(start, end-size)

            for slot in range(s, s+size):

                # ===============================
                # GWO voting (alpha beta delta)
                # ===============================

                kandidat_mapel = [
                    alpha_mapel[kelas,slot],
                    beta_mapel[kelas,slot],
                    delta_mapel[kelas,slot]
                ]

                kandidat_guru = [
                    alpha_guru[kelas,slot],
                    beta_guru[kelas,slot],
                    delta_guru[kelas,slot]
                ]

                # majority vote
                mapel_vote = Counter(kandidat_mapel).most_common(1)[0][0]
                guru_vote  = Counter(kandidat_guru).most_common(1)[0][0]

                # eksplorasi jika a besar
                if random.random() < a/2:
                    mapel[kelas,slot] = mapel_vote
                    guru[kelas,slot]  = guru_vote

                # eksplorasi random
                else:
                    tingkatan = kelasTingkatan[kelasIds[kelas]]
                    guruValid = guruValidMapel.get((mapel_vote, tingkatan))

                    if guruValid:
                        mapel[kelas,slot] = mapel_vote
                        guru[kelas,slot]  = random.choice(guruValid)

        # ===============================
        # local improvement swap
        # ===============================

        if random.random() < 0.5:

            kelas = random.randint(0,JUMLAH_KELAS-1)
            slot1 = random.randint(0,SLOT_PER_KELAS-1)
            slot2 = random.randint(0,SLOT_PER_KELAS-1)

            mapel[kelas,slot1], mapel[kelas,slot2] = mapel[kelas,slot2], mapel[kelas,slot1]
            guru[kelas,slot1],  guru[kelas,slot2]  = guru[kelas,slot2],  guru[kelas,slot1]

        # repair constraint
        mapel, guru = repairOperator(mapel, guru)

        # ===============================
        # accept only if better
        # ===============================

        oldFitness = hasilPop[i]["fitness"]

        evaluasi = evaluasiIndividu(mapel, guru, maxJamGuru)
        newFitness = hitungFitness(evaluasi)

        if newFitness <= oldFitness:
            new_pop_mapel.append(mapel)
            new_pop_guru.append(guru)
        else:
            new_pop_mapel.append(pop_mapel[i])
            new_pop_guru.append(pop_guru[i])

    return np.array(new_pop_mapel), np.array(new_pop_guru)

# MAIN

In [380]:
def runGA():

    # Membuat populasi awal jadwal mapel dan guru
    pop_mapel, pop_guru = generatePopulation(POPULASI)

    # Mengevaluasi seluruh populasi awal
    hasilPop = evaluasiPopulasi(pop_mapel, pop_guru, maxJamGuru)

    # Menyimpan fitness terbaik global
    bestFitness = float("inf")

    # Menyimpan log perkembangan fitness
    log_data = []

    # Iterasi utama algoritma
    for gen in range(ITERATION):

        # laoding efffect nang kene

        # Menyimpan populasi baru hasil GA
        new_pop_mapel = []
        new_pop_guru  = []

        # ==========================
        # PROSES GA
        # ==========================

        # Membuat populasi baru sampai jumlah individu terpenuhi
        while len(new_pop_mapel) < POPULASI:

            # Memilih parent menggunakan tournament selection
            p1_mapel, p1_guru = tournamentSelection(pop_mapel, pop_guru, hasilPop)
            p2_mapel, p2_guru = tournamentSelection(pop_mapel, pop_guru, hasilPop)

            # Jika memenuhi probabilitas crossover
            if random.random() < CROSSOVER_PROB:

                # Melakukan class-based crossover
                c1_mapel, c1_guru, c2_mapel, c2_guru = crossover(
                    p1_mapel, p1_guru, p2_mapel, p2_guru
                )

            else:

                # Jika tidak crossover, anak adalah copy dari parent
                c1_mapel, c1_guru = p1_mapel.copy(), p1_guru.copy()
                c2_mapel, c2_guru = p2_mapel.copy(), p2_guru.copy()

            # Mutasi anak pertama jika memenuhi probabilitas mutasi
            if random.random() < MUTATION_PROB:
                c1_mapel, c1_guru = mutasi(c1_mapel, c1_guru)

            # Mutasi anak kedua
            if random.random() < MUTATION_PROB:
                c2_mapel, c2_guru = mutasi(c2_mapel, c2_guru)

            # Repair untuk memperbaiki pelanggaran constraint
            # c1_mapel, c1_guru = repairOperator(c1_mapel, c1_guru)
            # c2_mapel, c2_guru = repairOperator(c2_mapel, c2_guru)

            # Menambahkan anak pertama ke populasi baru
            new_pop_mapel.append(c1_mapel)
            new_pop_guru.append(c1_guru)

            # Menambahkan anak kedua jika populasi belum penuh
            if len(new_pop_mapel) < POPULASI:
                new_pop_mapel.append(c2_mapel)
                new_pop_guru.append(c2_guru)

        # Mengganti populasi lama dengan populasi baru
        pop_mapel = np.array(new_pop_mapel)
        pop_guru  = np.array(new_pop_guru)

        # Mengevaluasi populasi baru
        hasilPop = evaluasiPopulasi(pop_mapel, pop_guru, maxJamGuru)

        # ==========================
        # PROSESS GWO
        # ==========================

        # Setiap 20 generasi dilakukan fase optimasi GWO
        # if gen % 20 == 0 and gen != 0:

        #     print("----- GWO PHASE -----")

        #     # Melakukan iterasi GWO sebanyak 20 kali
        #     for iter_gwo in range(20):

        #         # Update posisi populasi menggunakan Grey Wolf Optimizer
        #         pop_mapel, pop_guru = runGWO(pop_mapel, pop_guru, hasilPop)

        #         # Evaluasi kembali populasi hasil GWO
        #         hasilPop = evaluasiPopulasi(pop_mapel, pop_guru, maxJamGuru)

        #         # Mengambil fitness terbaik dalam populasi
        #         best_gwo = min(x["fitness"] for x in hasilPop)

        #         # Menghitung rata-rata fitness populasi
        #         avg_gwo  = sum(x["fitness"] for x in hasilPop) / len(hasilPop)

        #         # Menyimpan log GWO
        #         log_data.append([gen, "GWO", best_gwo, avg_gwo])

        #         print("GWO Iter", iter_gwo,"Best:", best_gwo,"Avg:", round(avg_gwo))

        # ==========================
        # BEST FITNESS
        # ==========================

        # Mengambil individu terbaik dalam populasi
        best = min(hasilPop, key=lambda x: x["fitness"])

        # Menghitung rata-rata fitness populasi
        avg = np.mean([x['fitness'] for x in hasilPop])

        # Update best global fitness jika ditemukan yang lebih baik
        if best["fitness"] < bestFitness:
            bestFitness = best["fitness"]

        # Menyimpan log GA
        log_data.append([gen,"GA", bestFitness, avg])

        # Menampilkan progress generasi
        print("Generasi", gen, "| Best Fitness:", bestFitness, "| avg:", avg)

    # ==========================
    # OUTPUT
    # ==========================

    # Menentukan path file untuk menyimpan log
    path_file = os.path.join("../hasil/record", RECORD_EXPORT)

    # Menyimpan log fitness ke file CSV
    with open(path_file, mode="w", newline="") as file:

        writer = csv.writer(file)

        # Header kolom
        writer.writerow(["iterasi", "method","best", "avg"])

        # Data log
        writer.writerows(log_data)

    # Mengembalikan individu terbaik
    return best

### Main Call

In [381]:
best = runGA()

print("Best Fitness:", best["fitness"])
print("Detail Evaluasi:")
print(best["evaluasi"])

Generasi 0 | Best Fitness: 19635 | avg: 21357.7
Generasi 1 | Best Fitness: 18855 | avg: 20210.4
Generasi 2 | Best Fitness: 18300 | avg: 19673.8
Generasi 3 | Best Fitness: 18220 | avg: 18974.4
Generasi 4 | Best Fitness: 18140 | avg: 18461.1
Generasi 5 | Best Fitness: 17980 | avg: 18206.7
Generasi 6 | Best Fitness: 17960 | avg: 18088.8
Generasi 7 | Best Fitness: 17725 | avg: 17989.6
Generasi 8 | Best Fitness: 17645 | avg: 17861.2
Generasi 9 | Best Fitness: 17555 | avg: 17729.8
Generasi 10 | Best Fitness: 17465 | avg: 17642.1
Generasi 11 | Best Fitness: 17375 | avg: 17524.1
Generasi 12 | Best Fitness: 17285 | avg: 17419.2
Generasi 13 | Best Fitness: 17205 | avg: 17342.6
Generasi 14 | Best Fitness: 17045 | avg: 17264.8
Generasi 15 | Best Fitness: 16975 | avg: 17146.2
Generasi 16 | Best Fitness: 16815 | avg: 17038.8
Generasi 17 | Best Fitness: 16725 | avg: 16911.0
Generasi 18 | Best Fitness: 16635 | avg: 16810.6
Generasi 19 | Best Fitness: 16545 | avg: 16686.1
Generasi 20 | Best Fitness: 16

In [382]:
kelas = 0
ind = 0

print("Mapel kelas", kelas+1)
print(pop_mapel[ind, kelas])

print("Guru kelas", kelas+1)
print(pop_guru[ind, kelas])

Mapel kelas 1
[ 5  5  6  6  6  5  5  4  4  2  2  3  3  3 12 12 12 11 11  8  8  7  7  7
  1  1 13 13  4  4  3  3  9  9 10 10]
Guru kelas 1
[48 48 44 44 44 48 48 47 47 33 33 36 36 36 55 55 55 43 43  5  5 21 21 21
 39 39 53 53 47 47 36 36 38 38 47 47]


In [383]:
best_index = best["index"]

best_mapel = pop_mapel[best_index]
best_guru  = pop_guru[best_index]

In [384]:
# print(best_mapel[0])
# print(type(best_mapel[0]))

---
# EXPORT FITNESS (OUTPUT)

In [385]:
# path_file = os.path.join("../hasil/bestFitness", FITNESS_EXPORT)

# with open(path_file, mode="w", newline="") as file:

#     writer = csv.writer(file)

#     writer.writerow(["kelas","slot","mapel","guru"])

#     for kelas in range(len(best_mapel)):

#         for slot in range(len(best_mapel[kelas])):

#             mapel = best_mapel[kelas][slot]
#             guru  = best_guru[kelas][slot]

#             writer.writerow([
#                 kelas + 1,
#                 slot+ 1,
#                 mapel,
#                 guru
#             ])

---
# Example

In [386]:
kelas_idx = 1

data = []

for slot in range(SLOT_PER_KELAS):

    mapel_id = best_mapel[kelas_idx, slot]
    guru_id  = best_guru[kelas_idx, slot]
    hari     = slotHari[slot]

    data.append({
        "Slot": slot + 1,
        "Hari": hari,
        "Mapel": mapel_id,
        "Guru": guru_id
    })

df = pd.DataFrame(data)

print(df)

    Slot  Hari  Mapel  Guru
0      1     1      4    47
1      2     1      4    47
2      3     1      1    31
3      4     1      1    31
4      5     1     11    43
5      6     1     11    43
6      7     1      8     5
7      8     1      8     5
8      9     2      9    38
9     10     2      9    38
10    11     2      4    47
11    12     2      4    47
12    13     2      7    21
13    14     2      7    21
14    15     2      7    21
15    16     2     10    48
16    17     3     10    48
17    18     3      3    29
18    19     3      3    29
19    20     3      3    29
20    21     3      3    29
21    22     3      3    29
22    23     3      5    48
23    24     3      5    48
24    25     4      6     3
25    26     4      6     3
26    27     4      6     3
27    28     4     13    53
28    29     4     13    53
29    30     4      2    33
30    31     4      2    33
31    32     5     12    51
32    33     5     12    51
33    34     5     12    51
34    35     5      

---

# Bimbingan 12/03/2026


mulai pengerjaan jurnal (UI opsional)  
buku TA  
apakah boleh fitness != 0  
UI nya hanya crud via spreadsheet, yang penting page generate jadwal   
dokumentasikan iterasi untuk diagram garis  
mutasi dan crossover  
minus GWO  
bikin template excell UI untuk CRUD  

===================================================  
before (sistem akbar)  
21680 awal data masuk  

after (sistem akbar)  
2480 akhir stelah optimasi  

hal manual vs digitalize = apakah jadwal lebih oke atau tidak?  
perbandingan runtime GA single vs GA-GWO  

SK keluar tgl nya februari  

kemudian bisa ngisi bimgingna online  

---
# catatan saya
Generate Population

FOR generation

    Evaluasi Fitness

    GA Operator
        Selection
        Crossover
        Mutation

    GWO Local Search
        update solusi menuju alpha beta delta

    Repair Operator

END

# CRUD 
# halaman generate jadwal ()

# /master

# Bimbingan 17/03

tampilkan ebfore after evaluasi

pertanyaan yang masih ada = optimasi bisa jadi ke oke an jadwal, parameter keberhasilan (detail evaluasi)

introduction = tambahkan urgensi objek

parameter dikasih tau di methodology

introduction = kasih spill jadwal

metolodogi = tambahkan flowchart dari data masuk hingga settigan GA dan GWO hingga output

hasil pertama = jadwal seperti apa | tabel detail evaluasi | runtime

hasil = grafik ditambahkan di akhir

convergence analysis = convergence handle

membandingkan GA biasa | Guided GA | guided GA GWO = skenario 2